# P10.6-AI — Notebook 57: preflight de estrechamiento foraminal

Prepara la segunda línea RSNA/LumbarDISC del PFI: **estrechamiento foraminal neural izquierdo y derecho**, por nivel lumbar, usando series **Sagittal T1**.

Este notebook no entrena modelos ni accede al test oficial. Audita etiquetas, lateralidad, niveles, disponibilidad de secuencias y coordenadas; luego exporta un manifiesto candidato para que el Notebook 58 realice el split interno por `study_id`.

`humanReviewRequired=true` · `notClinicalDiagnosis=true` · `officialTestAccessed=false`


## Antes de ejecutar

- Runtime recomendado: **Colab estándar con CPU**. No requiere GPU.
- Autorizar acceso a Google Drive.
- Tener una cuenta de Kaggle que haya aceptado las reglas de `rsna-2024-lumbar-spine-degenerative-classification`.
- Tener disponible `KAGGLE_API_TOKEN` cuando se solicite. El valor se ingresa oculto y se elimina del entorno.
- Espacio local recomendado: al menos **1 GiB libre**. Solo se descargan tres CSV de entrenamiento, no las imágenes.
- El repositorio es público: no se necesita token de GitHub para clonarlo.


In [1]:
# 1) Dependencias mínimas
from __future__ import annotations

import importlib.util
import subprocess
import sys

REQUIRED = {
    "numpy": "numpy",
    "pandas": "pandas",
    "yaml": "pyyaml",
    "kaggle": "kaggle",
}

missing = [
    package
    for module, package in REQUIRED.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        *missing,
    ])

print({"python": sys.version.split()[0], "installedNow": missing})


{'python': '3.12.13', 'installedNow': []}


In [2]:
# 2) Montar Drive y clonar/actualizar la rama de trabajo
import json
import os
import shutil
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
from google.colab import drive  # type: ignore

drive.mount("/content/drive", force_remount=False)

REPO_URL = (
    "https://github.com/EnzoAA004/"
    "PFI_MVPTest_Enzo_AImodule.git"
)
REPO_ROOT = Path("/content/PFI_MVPTest_Enzo_AImodule")
REPO_REF = "enzo/p10-6-ai-rsna-findings"

if not (REPO_ROOT / ".git").exists():
    subprocess.check_call([
        "git",
        "clone",
        "--branch",
        REPO_REF,
        "--single-branch",
        REPO_URL,
        str(REPO_ROOT),
    ])
else:
    subprocess.check_call([
        "git", "fetch", "origin", REPO_REF
    ], cwd=REPO_ROOT)
    subprocess.check_call([
        "git", "checkout", REPO_REF
    ], cwd=REPO_ROOT)
    subprocess.check_call([
        "git", "pull", "--ff-only", "origin", REPO_REF
    ], cwd=REPO_ROOT)

sys.path.insert(0, str(REPO_ROOT / "ai_service"))

from pfi_ai_service.training.rsna_preflight import (
    LEVELS,
    atomic_write_json,
    atomic_write_text,
    normalize_condition,
    normalize_level,
    normalize_series_description,
    normalize_severity,
    parse_label_column,
    sha256_file,
)

REPO_SHA = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_ROOT,
    text=True,
).strip()
print({"repoRef": REPO_REF, "repoSha": REPO_SHA})


Mounted at /content/drive
{'repoRef': 'enzo/p10-6-ai-rsna-findings', 'repoSha': 'e90f9d8c70c08b81d9020769030a0e55ee158078'}


In [3]:
# 3) Rutas, alcance y gates del preflight
PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
OUTPUT_ROOT = (
    PFI_ROOT
    / "results"
    / "P10_6_rsna_findings"
    / "notebook57_foraminal_preflight"
)
LOCAL_ROOT = Path("/content/RSNA_LUMBAR_DISC_FORAMINAL_PREFLIGHT")
DOWNLOAD_ROOT = Path("/content/rsna_foraminal_csv_download")
COMPETITION = (
    "rsna-2024-lumbar-spine-degenerative-classification"
)
REQUIRED_CSVS = (
    "train.csv",
    "train_label_coordinates.csv",
    "train_series_descriptions.csv",
)
SIDES = ("left", "right")
SEVERITY_CODE = {
    "normal_mild": 0,
    "moderate": 1,
    "severe": 2,
}
MIN_T1_STUDY_COVERAGE = 0.95
MIN_USABLE_COORDINATE_COVERAGE = 0.97
MAX_UNKNOWN_LABEL_RATE = 0.005

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

total, used, free = shutil.disk_usage("/content")
print({
    "outputRoot": str(OUTPUT_ROOT),
    "localRoot": str(LOCAL_ROOT),
    "runtimeFreeGiB": round(free / 1024**3, 2),
    "gpuRequired": False,
})
if free < 1024**3:
    raise RuntimeError(
        "Espacio local insuficiente: se requiere al menos 1 GiB libre."
    )


{'outputRoot': '/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings/notebook57_foraminal_preflight', 'localRoot': '/content/RSNA_LUMBAR_DISC_FORAMINAL_PREFLIGHT', 'runtimeFreeGiB': 87.67, 'gpuRequired': False}


## Descarga CSV-only

Se reutilizan CSV ya presentes en el runtime o en Drive. Si faltan, se descargan individualmente desde Kaggle. No se descarga `train_images/`, `test_images/` ni ningún archivo del test oficial.


In [4]:
# 4) Reutilizar CSV locales o descargarlos individualmente desde Kaggle
import getpass
import zipfile

def csvs_ready(root: Path) -> bool:
    return all(
        (root / name).is_file() and (root / name).stat().st_size > 0
        for name in REQUIRED_CSVS
    )


def copy_from_existing_runtime() -> bool:
    candidate_roots = [
        Path("/content/RSNA_LUMBAR_DISC"),
        PFI_ROOT / "data" / "RSNA_LUMBAR_DISC",
    ]
    copied = []
    for candidate_root in candidate_roots:
        if not csvs_ready(candidate_root):
            continue
        for name in REQUIRED_CSVS:
            destination = LOCAL_ROOT / name
            if not destination.exists():
                shutil.copy2(candidate_root / name, destination)
                copied.append(name)
        print({
            "csvSource": str(candidate_root),
            "copied": copied,
        })
        return csvs_ready(LOCAL_ROOT)
    return False


def safe_extract_named_csv(archive_path: Path, expected_name: str) -> None:
    root_resolved = LOCAL_ROOT.resolve()
    matches = 0
    with zipfile.ZipFile(archive_path, "r") as archive:
        for member in archive.infolist():
            if member.is_dir():
                continue
            normalized = PurePosixPath(member.filename)
            if normalized.is_absolute() or ".." in normalized.parts:
                raise RuntimeError(
                    f"Ruta insegura dentro del ZIP: {member.filename!r}"
                )
            if normalized.name != expected_name:
                continue
            destination = (LOCAL_ROOT / expected_name).resolve()
            if root_resolved not in destination.parents:
                raise RuntimeError("Extracción fuera del root permitido.")
            with archive.open(member) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target, length=1024 * 1024)
            matches += 1
    if matches != 1:
        raise RuntimeError(
            f"Se esperó un único {expected_name} dentro de {archive_path.name}; "
            f"encontrados={matches}."
        )


def download_csvs_individually(kaggle_token: str) -> None:
    DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
    executable = shutil.which("kaggle") or str(
        Path(sys.executable).parent / "kaggle"
    )
    os.environ["KAGGLE_API_TOKEN"] = kaggle_token
    try:
        for name in REQUIRED_CSVS:
            subprocess.check_call([
                executable,
                "competitions",
                "download",
                "-c", COMPETITION,
                "-f", name,
                "-p", str(DOWNLOAD_ROOT),
                "--force",
            ])

            direct = DOWNLOAD_ROOT / name
            archive_candidates = [
                DOWNLOAD_ROOT / f"{name}.zip",
                *sorted(DOWNLOAD_ROOT.glob(f"*{name}*.zip")),
            ]

            if direct.is_file() and direct.stat().st_size > 0:
                shutil.move(str(direct), str(LOCAL_ROOT / name))
            else:
                archive = next(
                    (
                        path
                        for path in archive_candidates
                        if path.is_file() and path.stat().st_size > 0
                    ),
                    None,
                )
                if archive is None:
                    raise RuntimeError(
                        f"Kaggle no produjo un archivo utilizable para {name}."
                    )
                safe_extract_named_csv(archive, name)
    finally:
        os.environ.pop("KAGGLE_API_TOKEN", None)
        shutil.rmtree(DOWNLOAD_ROOT, ignore_errors=True)


if not csvs_ready(LOCAL_ROOT):
    copy_from_existing_runtime()

if not csvs_ready(LOCAL_ROOT):
    token = getpass.getpass(
        "Pegá tu KAGGLE_API_TOKEN (no se mostrará): "
    ).strip()
    if not token:
        raise RuntimeError("No se ingresó token de Kaggle.")
    try:
        download_csvs_individually(token)
    finally:
        token = ""

if not csvs_ready(LOCAL_ROOT):
    raise RuntimeError("Los tres CSV de entrenamiento no quedaron disponibles.")

print({
    name: {
        "bytes": int((LOCAL_ROOT / name).stat().st_size),
        "sha256": sha256_file(LOCAL_ROOT / name),
    }
    for name in REQUIRED_CSVS
})


{'csvSource': '/content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC', 'copied': ['train.csv', 'train_label_coordinates.csv', 'train_series_descriptions.csv']}
{'train.csv': {'bytes': 568972, 'sha256': 'f0c9e06486bcddcd83b1ee1d95ab9db8e028d7c3c9fcbada950f0b8e4d828528'}, 'train_label_coordinates.csv': {'bytes': 4642067, 'sha256': '416fb434b5bdbf69814210d03dd791dff5ef9a010bac004f5f68e8b79e852635'}, 'train_series_descriptions.csv': {'bytes': 213642, 'sha256': 'bf8cc1aa55e4b5536b2f0fa8a060fc1912faeb65f9ba422d9d91b92a947b1c33'}}


In [5]:
# 5) Cargar y validar los CSV train-only
train = pd.read_csv(
    LOCAL_ROOT / "train.csv",
    dtype={"study_id": str},
)
coordinates = pd.read_csv(
    LOCAL_ROOT / "train_label_coordinates.csv",
    dtype={
        "study_id": str,
        "series_id": str,
        "instance_number": "Int64",
    },
)
series = pd.read_csv(
    LOCAL_ROOT / "train_series_descriptions.csv",
    dtype={
        "study_id": str,
        "series_id": str,
        "series_description": str,
    },
)

required_train = {"study_id"}
required_coordinates = {
    "study_id", "series_id", "instance_number",
    "condition", "level", "x", "y",
}
required_series = {
    "study_id", "series_id", "series_description",
}
for name, frame, required in [
    ("train", train, required_train),
    ("coordinates", coordinates, required_coordinates),
    ("series", series, required_series),
]:
    missing_columns = sorted(required - set(frame.columns))
    if missing_columns:
        raise RuntimeError(
            f"{name} no contiene columnas requeridas: {missing_columns}"
        )

if train["study_id"].duplicated().any():
    raise RuntimeError("train.csv contiene study_id duplicados.")
if series[["study_id", "series_id"]].duplicated().any():
    raise RuntimeError(
        "train_series_descriptions.csv contiene claves duplicadas."
    )

print({
    "studies": int(train["study_id"].nunique()),
    "series": int(series["series_id"].nunique()),
    "coordinateRows": int(len(coordinates)),
    "officialTestAccessed": False,
})


{'studies': 1975, 'series': 6294, 'coordinateRows': 48692, 'officialTestAccessed': False}


In [6]:
# 6) Construir etiquetas foraminales izquierda/derecha por nivel
target_columns = []
for column in train.columns:
    if column == "study_id":
        continue
    condition, level = parse_label_column(str(column))
    if condition in {
        "neural_foraminal_narrowing_left",
        "neural_foraminal_narrowing_right",
    } and level in LEVELS:
        side = condition.rsplit("_", 1)[-1]
        target_columns.append((str(column), condition, side, level))

expected_pairs = {
    (side, level)
    for side in SIDES
    for level in LEVELS
}
observed_pairs = {(side, level) for _, _, side, level in target_columns}
if observed_pairs != expected_pairs:
    raise RuntimeError({
        "missingTargets": sorted(expected_pairs - observed_pairs),
        "unexpectedTargets": sorted(observed_pairs - expected_pairs),
    })

label_rows = []
for column, condition, side, level in target_columns:
    frame = train[["study_id", column]].copy()
    frame = frame.rename(columns={column: "severity_raw"})
    frame["condition"] = condition
    frame["side"] = side
    frame["level"] = level
    frame["rsna_label_column"] = column
    frame["severity"] = frame["severity_raw"].map(
        normalize_severity
    )
    label_rows.append(frame)

labels = pd.concat(label_rows, ignore_index=True)
labels["severity_code"] = labels["severity"].map(SEVERITY_CODE)
unknown_label_count = int(labels["severity"].isna().sum())
unknown_label_rate = unknown_label_count / max(len(labels), 1)

expected_rows = int(train["study_id"].nunique()) * len(expected_pairs)
if len(labels) != expected_rows:
    raise RuntimeError(
        f"Cantidad de etiquetas inesperada: {len(labels)} != {expected_rows}."
    )
if labels[["study_id", "side", "level"]].duplicated().any():
    raise RuntimeError("Hay claves study-side-level duplicadas.")

label_distribution = (
    labels
    .groupby(["side", "level", "severity"], dropna=False)
    .size()
    .reset_index(name="count")
)
label_distribution["percent_within_side_level"] = (
    label_distribution["count"]
    / label_distribution.groupby(["side", "level"])["count"]
      .transform("sum")
    * 100.0
)

print({
    "targetColumns": len(target_columns),
    "labelRows": int(len(labels)),
    "unknownLabelCount": unknown_label_count,
    "unknownLabelRate": round(unknown_label_rate, 6),
})
display(label_distribution)


{'targetColumns': 10, 'labelRows': 19750, 'unknownLabelCount': 50, 'unknownLabelRate': 0.002532}


,side,level,severity,count,percent_within_side_level
0,left,L1-L2,moderate,63,3.189873
1,left,L1-L2,normal_mild,1908,96.607595
2,left,L1-L2,severe,2,0.101266
3,left,L1-L2,NaN,2,0.101266
4,left,L2-L3,moderate,171,8.658228
5,left,L2-L3,normal_mild,1791,90.683544
6,left,L2-L3,severe,11,0.556962
7,left,L2-L3,NaN,2,0.101266
8,left,L3-L4,moderate,411,20.810127
9,left,L3-L4,normal_mild,1522,77.063291


In [7]:
# 7) Inventariar series Sagittal T1 sin abrir imágenes
series_normalized = series.copy()
description_parts = series_normalized["series_description"].map(
    normalize_series_description
)
series_normalized["plane"] = description_parts.map(
    lambda item: item["plane"]
)
series_normalized["sequence"] = description_parts.map(
    lambda item: item["sequence"]
)
series_normalized["sequence_category"] = description_parts.map(
    lambda item: item["sequenceCategory"]
)
sagittal_t1 = series_normalized.loc[
    series_normalized["sequence_category"].eq("sagittal_t1")
].copy()

study_t1 = (
    sagittal_t1.groupby("study_id")
    .agg(
        sagittal_t1_series_count=("series_id", "nunique"),
        sagittal_t1_series_ids=(
            "series_id",
            lambda values: "|".join(sorted(set(map(str, values)))),
        ),
    )
    .reset_index()
)
study_t1 = train[["study_id"]].merge(
    study_t1,
    on="study_id",
    how="left",
    validate="one_to_one",
)
study_t1["sagittal_t1_series_count"] = (
    study_t1["sagittal_t1_series_count"].fillna(0).astype(int)
)
study_t1["sagittal_t1_series_ids"] = (
    study_t1["sagittal_t1_series_ids"].fillna("")
)
study_t1["has_sagittal_t1"] = (
    study_t1["sagittal_t1_series_count"] > 0
)
t1_study_coverage = float(study_t1["has_sagittal_t1"].mean())

print({
    "sagittalT1Series": int(len(sagittal_t1)),
    "studiesWithSagittalT1": int(
        study_t1["has_sagittal_t1"].sum()
    ),
    "studyCoverage": round(t1_study_coverage, 6),
    "studiesWithMultipleT1Series": int(
        (study_t1["sagittal_t1_series_count"] > 1).sum()
    ),
})


{'sagittalT1Series': 1980, 'studiesWithSagittalT1': 1973, 'studyCoverage': 0.998987, 'studiesWithMultipleT1Series': 7}


In [8]:
# 8) Normalizar y validar coordenadas foraminales
coordinate_frame = coordinates.copy()
coordinate_frame["condition_normalized"] = coordinate_frame[
    "condition"
].map(normalize_condition)
coordinate_frame["level_normalized"] = coordinate_frame["level"].map(
    normalize_level
)
coordinate_frame = coordinate_frame.loc[
    coordinate_frame["condition_normalized"].isin({
        "neural_foraminal_narrowing_left",
        "neural_foraminal_narrowing_right",
    })
].copy()
coordinate_frame["side"] = coordinate_frame[
    "condition_normalized"
].str.rsplit("_", n=1).str[-1]
coordinate_frame["level"] = coordinate_frame["level_normalized"]

coordinate_frame = coordinate_frame.merge(
    series_normalized[[
        "study_id", "series_id", "series_description",
        "sequence_category",
    ]],
    on=["study_id", "series_id"],
    how="left",
    validate="many_to_one",
)
coordinate_frame["coordinate_on_sagittal_t1"] = (
    coordinate_frame["sequence_category"].eq("sagittal_t1")
)
coordinate_frame["coordinate_numeric"] = (
    coordinate_frame[["instance_number", "x", "y"]]
    .notna()
    .all(axis=1)
)
coordinate_frame["coordinate_nonnegative"] = (
    coordinate_frame["x"].fillna(-1).ge(0)
    & coordinate_frame["y"].fillna(-1).ge(0)
    & coordinate_frame["instance_number"].fillna(-1).ge(0)
)
coordinate_frame["valid_foraminal_coordinate"] = (
    coordinate_frame["coordinate_on_sagittal_t1"]
    & coordinate_frame["coordinate_numeric"]
    & coordinate_frame["coordinate_nonnegative"]
    & coordinate_frame["level"].isin(LEVELS)
)

valid_coordinates = coordinate_frame.loc[
    coordinate_frame["valid_foraminal_coordinate"]
].copy()
valid_coordinates = valid_coordinates.sort_values([
    "study_id", "side", "level", "series_id",
    "instance_number",
])
coordinate_counts = (
    valid_coordinates.groupby(["study_id", "side", "level"])
    .size()
    .reset_index(name="valid_coordinate_candidates")
)
selected_coordinates = (
    valid_coordinates
    .drop_duplicates(["study_id", "side", "level"], keep="first")
    [[
        "study_id", "side", "level", "series_id",
        "instance_number", "x", "y",
        "series_description",
    ]]
    .rename(columns={
        "series_id": "coordinate_series_id",
        "instance_number": "coordinate_instance_number",
        "x": "coordinate_x",
        "y": "coordinate_y",
        "series_description": "coordinate_series_description",
    })
)

duplicate_coordinate_candidates = valid_coordinates.merge(
    coordinate_counts.loc[
        coordinate_counts["valid_coordinate_candidates"] > 1
    ],
    on=["study_id", "side", "level"],
    how="inner",
)

print({
    "foraminalCoordinateRows": int(len(coordinate_frame)),
    "validSagittalT1Coordinates": int(len(valid_coordinates)),
    "duplicateCandidateRows": int(
        len(duplicate_coordinate_candidates)
    ),
})


{'foraminalCoordinateRows': 19719, 'validSagittalT1Coordinates': 19719, 'duplicateCandidateRows': 0}


In [9]:
# 9) Construir el manifiesto candidato study-side-level
manifest = labels.merge(
    study_t1,
    on="study_id",
    how="left",
    validate="many_to_one",
)
manifest = manifest.merge(
    coordinate_counts,
    on=["study_id", "side", "level"],
    how="left",
    validate="one_to_one",
)
manifest = manifest.merge(
    selected_coordinates,
    on=["study_id", "side", "level"],
    how="left",
    validate="one_to_one",
)
manifest["valid_coordinate_candidates"] = (
    manifest["valid_coordinate_candidates"].fillna(0).astype(int)
)
manifest["coordinate_status"] = np.select(
    [
        ~manifest["has_sagittal_t1"],
        manifest["valid_coordinate_candidates"].eq(0),
        manifest["valid_coordinate_candidates"].eq(1),
        manifest["valid_coordinate_candidates"].gt(1),
    ],
    [
        "missing_sagittal_t1",
        "missing_coordinate",
        "valid_unique",
        "valid_multiple",
    ],
    default="invalid",
)
manifest["usable_for_split"] = (
    manifest["severity"].notna()
    & manifest["has_sagittal_t1"]
    & manifest["valid_coordinate_candidates"].ge(1)
)
manifest["requires_coordinate_review"] = (
    manifest["valid_coordinate_candidates"].gt(1)
)
manifest["human_review_required"] = True
manifest["not_clinical_diagnosis"] = True
manifest["official_test_accessed"] = False

manifest = manifest.sort_values(
    ["study_id", "side", "level"]
).reset_index(drop=True)

usable_coordinate_coverage = float(
    manifest["usable_for_split"].mean()
)
exclusions = manifest.loc[~manifest["usable_for_split"]].copy()

coordinate_coverage = (
    manifest.groupby(["side", "level", "coordinate_status"])
    .size()
    .reset_index(name="count")
)
coordinate_coverage["percent_within_side_level"] = (
    coordinate_coverage["count"]
    / coordinate_coverage.groupby(["side", "level"])["count"]
      .transform("sum")
    * 100.0
)

print({
    "manifestRows": int(len(manifest)),
    "usableRows": int(manifest["usable_for_split"].sum()),
    "excludedRows": int(len(exclusions)),
    "usableCoordinateCoverage": round(
        usable_coordinate_coverage, 6
    ),
})
display(coordinate_coverage)


{'manifestRows': 19750, 'usableRows': 19689, 'excludedRows': 61, 'usableCoordinateCoverage': 0.996911}


,side,level,coordinate_status,count,percent_within_side_level
0,left,L1-L2,missing_coordinate,1,0.050633
1,left,L1-L2,missing_sagittal_t1,2,0.101266
2,left,L1-L2,valid_unique,1972,99.848101
3,left,L2-L3,missing_coordinate,1,0.050633
4,left,L2-L3,missing_sagittal_t1,2,0.101266
5,left,L2-L3,valid_unique,1972,99.848101
6,left,L3-L4,missing_coordinate,1,0.050633
7,left,L3-L4,missing_sagittal_t1,2,0.101266
8,left,L3-L4,valid_unique,1972,99.848101
9,left,L4-L5,missing_coordinate,1,0.050633


In [10]:
# 10) Resumen por estudio y soporte de clases
study_summary = (
    manifest.groupby("study_id")
    .agg(
        target_rows=("study_id", "size"),
        usable_rows=("usable_for_split", "sum"),
        severe_targets=("severity_code", lambda values: int((values == 2).sum())),
        moderate_targets=("severity_code", lambda values: int((values == 1).sum())),
        normal_mild_targets=("severity_code", lambda values: int((values == 0).sum())),
        sagittal_t1_series_count=("sagittal_t1_series_count", "first"),
        requires_coordinate_review=("requires_coordinate_review", "any"),
    )
    .reset_index()
)
study_summary["complete_for_all_targets"] = (
    study_summary["usable_rows"].eq(len(expected_pairs))
)

class_support = (
    manifest.loc[manifest["usable_for_split"]]
    .groupby(["side", "level"])["severity"]
    .nunique(dropna=True)
    .reset_index(name="n_severity_classes")
)
all_side_levels_have_three_classes = bool(
    len(class_support) == len(expected_pairs)
    and class_support["n_severity_classes"].eq(3).all()
)

display(study_summary.head())
display(class_support)


,study_id,target_rows,usable_rows,severe_targets,moderate_targets,normal_mild_targets,sagittal_t1_series_count,requires_coordinate_review,complete_for_all_targets
0,100206310,10,10,2,6,2,1,False,True
1,1002894806,10,10,1,4,5,1,False,True
2,1004726367,10,10,0,0,10,1,False,True
3,1008446160,10,10,0,1,9,1,False,True
4,1009445512,10,10,1,3,6,1,False,True


,side,level,n_severity_classes
0,left,L1-L2,3
1,left,L2-L3,3
2,left,L3-L4,3
3,left,L4-L5,3
4,left,L5-S1,3
5,right,L1-L2,3
6,right,L2-L3,3
7,right,L3-L4,3
8,right,L4-L5,3
9,right,L5-S1,3


In [11]:
# 11) Aplicar gates de aprobación para Notebook 58
gate_results = {
    "expectedTargetColumns": len(target_columns) == 10,
    "completeTargetRows": len(labels) == expected_rows,
    "unknownLabelRate": (
        unknown_label_rate <= MAX_UNKNOWN_LABEL_RATE
    ),
    "sagittalT1StudyCoverage": (
        t1_study_coverage >= MIN_T1_STUDY_COVERAGE
    ),
    "usableCoordinateCoverage": (
        usable_coordinate_coverage
        >= MIN_USABLE_COORDINATE_COVERAGE
    ),
    "allSideLevelsHaveThreeClasses": (
        all_side_levels_have_three_classes
    ),
    "noLabelKeyDuplicates": not labels[
        ["study_id", "side", "level"]
    ].duplicated().any(),
    "officialTestAccessed": False,
    "humanReviewRequired": True,
    "notClinicalDiagnosis": True,
}

approved = all([
    gate_results["expectedTargetColumns"],
    gate_results["completeTargetRows"],
    gate_results["unknownLabelRate"],
    gate_results["sagittalT1StudyCoverage"],
    gate_results["usableCoordinateCoverage"],
    gate_results["allSideLevelsHaveThreeClasses"],
    gate_results["noLabelKeyDuplicates"],
    gate_results["officialTestAccessed"] is False,
    gate_results["humanReviewRequired"],
    gate_results["notClinicalDiagnosis"],
])

print(json.dumps(gate_results, indent=2, ensure_ascii=False))
print({
    "status": (
        "APPROVED_FOR_NOTEBOOK_58"
        if approved
        else "PREFLIGHT_REVIEW_REQUIRED"
    )
})


{
  "expectedTargetColumns": true,
  "completeTargetRows": true,
  "unknownLabelRate": true,
  "sagittalT1StudyCoverage": true,
  "usableCoordinateCoverage": true,
  "allSideLevelsHaveThreeClasses": true,
  "noLabelKeyDuplicates": true,
  "officialTestAccessed": false,
  "humanReviewRequired": true,
  "notClinicalDiagnosis": true
}
{'status': 'APPROVED_FOR_NOTEBOOK_58'}


In [12]:
# 12) Exportar evidencia auditable a Google Drive
def atomic_write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.parent / f".{path.name}.tmp"
    try:
        frame.to_csv(temporary, index=False)
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()


source_hashes = {
    name: sha256_file(LOCAL_ROOT / name)
    for name in REQUIRED_CSVS
}

summary = {
    "schemaVersion": "pfi.rsna-foraminal-preflight.v1",
    "ticket": "P10.6-AI",
    "notebook": 57,
    "createdAtUtc": datetime.now(timezone.utc).isoformat(),
    "repoRef": REPO_REF,
    "repoSha": REPO_SHA,
    "dataset": "RSNA_LumbarDISC",
    "task": "neural_foraminal_narrowing",
    "sequence": "Sagittal T1",
    "sides": list(SIDES),
    "levels": list(LEVELS),
    "classes": list(SEVERITY_CODE),
    "nStudies": int(train["study_id"].nunique()),
    "targetRows": int(len(manifest)),
    "usableRows": int(manifest["usable_for_split"].sum()),
    "excludedRows": int((~manifest["usable_for_split"]).sum()),
    "unknownLabelCount": unknown_label_count,
    "unknownLabelRate": unknown_label_rate,
    "sagittalT1": {
        "series": int(len(sagittal_t1)),
        "studies": int(study_t1["has_sagittal_t1"].sum()),
        "studyCoverage": t1_study_coverage,
        "studiesWithMultipleSeries": int(
            (study_t1["sagittal_t1_series_count"] > 1).sum()
        ),
    },
    "coordinates": {
        "sourceRows": int(len(coordinate_frame)),
        "validRows": int(
            coordinate_frame["valid_foraminal_coordinate"].sum()
        ),
        "usableManifestCoverage": usable_coordinate_coverage,
        "duplicateValidCandidateRows": int(
            len(duplicate_coordinate_candidates)
        ),
        "statusCounts": {
            str(key): int(value)
            for key, value in (
                manifest["coordinate_status"]
                .value_counts(dropna=False)
                .to_dict()
                .items()
            )
        },
    },
    "gates": {
        "minimumSagittalT1StudyCoverage": (
            MIN_T1_STUDY_COVERAGE
        ),
        "minimumUsableCoordinateCoverage": (
            MIN_USABLE_COORDINATE_COVERAGE
        ),
        "maximumUnknownLabelRate": (
            MAX_UNKNOWN_LABEL_RATE
        ),
    },
    "gateResults": gate_results,
    "approved": approved,
    "nextNotebook": 58 if approved else None,
    "sourceCsvSha256": source_hashes,
    "governance": {
        "commercialUse": False,
        "humanReviewRequired": True,
        "notClinicalDiagnosis": True,
        "autonomousDiagnosis": False,
        "officialTestAccessed": False,
        "predictionStatus": "not_applicable_preflight",
    },
    "limitations": [
        "No model was trained.",
        "No DICOM pixels were downloaded or inspected.",
        "Coordinate validity is structural and sequence-based; "
        "pixel-bound checks are deferred to the training subset.",
        "Rows without a usable Sagittal T1 coordinate must be "
        "excluded or manually reviewed before training.",
        "The output is for assisted research and not clinical diagnosis.",
    ],
}

paths = {
    "manifest": OUTPUT_ROOT / "foraminal_candidate_manifest.csv",
    "labelDistribution": (
        OUTPUT_ROOT / "foraminal_label_distribution.csv"
    ),
    "coordinateCoverage": (
        OUTPUT_ROOT / "foraminal_coordinate_coverage.csv"
    ),
    "t1Inventory": (
        OUTPUT_ROOT / "sagittal_t1_series_inventory.csv"
    ),
    "studySummary": (
        OUTPUT_ROOT / "foraminal_study_summary.csv"
    ),
    "exclusions": (
        OUTPUT_ROOT / "foraminal_exclusions.csv"
    ),
    "duplicateCoordinates": (
        OUTPUT_ROOT
        / "foraminal_duplicate_coordinate_candidates.csv"
    ),
    "summary": OUTPUT_ROOT / "foraminal_preflight_summary.json",
    "report": OUTPUT_ROOT / "foraminal_preflight_report.md",
}

atomic_write_csv(paths["manifest"], manifest)
atomic_write_csv(paths["labelDistribution"], label_distribution)
atomic_write_csv(paths["coordinateCoverage"], coordinate_coverage)
atomic_write_csv(paths["t1Inventory"], sagittal_t1)
atomic_write_csv(paths["studySummary"], study_summary)
atomic_write_csv(paths["exclusions"], exclusions)
atomic_write_csv(
    paths["duplicateCoordinates"],
    duplicate_coordinate_candidates,
)
atomic_write_json(paths["summary"], summary)

report_lines = [
    "# P10.6-AI — Preflight foraminal RSNA",
    "",
    "Preflight train-only para estrechamiento foraminal neural "
    "izquierdo y derecho sobre Sagittal T1.",
    "",
    f"- Estado: `{'APPROVED_FOR_NOTEBOOK_58' if approved else 'PREFLIGHT_REVIEW_REQUIRED'}`",
    f"- Estudios: {summary['nStudies']}",
    f"- Filas objetivo: {summary['targetRows']}",
    f"- Filas utilizables: {summary['usableRows']}",
    f"- Cobertura Sagittal T1 por estudio: {t1_study_coverage:.4f}",
    f"- Cobertura de coordenadas utilizables: {usable_coordinate_coverage:.4f}",
    f"- Etiquetas desconocidas: {unknown_label_count}",
    f"- Test oficial accedido: false",
    "",
    "## Gates",
    "",
]
for name, result in gate_results.items():
    report_lines.append(f"- {name}: `{str(result).lower()}`")

report_lines.extend([
    "",
    "## Próximo paso",
    "",
    (
        "Notebook 58: split interno por `study_id`, con "
        "estratificación multilabel aproximada y reserva del test "
        "interno."
        if approved
        else
        "Revisar `foraminal_exclusions.csv` y los gates fallidos "
        "antes de crear el split."
    ),
    "",
    "## Límites",
    "",
    "- No se entrenó ningún modelo.",
    "- No se descargaron imágenes DICOM.",
    "- No constituye diagnóstico clínico.",
    "- Toda salida futura requiere revisión profesional.",
    "",
])

atomic_write_text(
    paths["report"],
    "\n".join(report_lines),
)

output_hashes = {
    name: sha256_file(path)
    for name, path in paths.items()
    if path.is_file() and name != "summary"
}
summary["outputSha256"] = output_hashes
atomic_write_json(paths["summary"], summary)

print({
    "outputRoot": str(OUTPUT_ROOT),
    "outputs": sorted(path.name for path in paths.values()),
    "status": (
        "APPROVED_FOR_NOTEBOOK_58"
        if approved
        else "PREFLIGHT_REVIEW_REQUIRED"
    ),
})


{'outputRoot': '/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings/notebook57_foraminal_preflight', 'outputs': ['foraminal_candidate_manifest.csv', 'foraminal_coordinate_coverage.csv', 'foraminal_duplicate_coordinate_candidates.csv', 'foraminal_exclusions.csv', 'foraminal_label_distribution.csv', 'foraminal_preflight_report.md', 'foraminal_preflight_summary.json', 'foraminal_study_summary.csv', 'sagittal_t1_series_inventory.csv'], 'status': 'APPROVED_FOR_NOTEBOOK_58'}


In [13]:
# 13) Gate final de cierre
required_outputs = list(paths.values())
missing_outputs = [
    str(path) for path in required_outputs if not path.is_file()
]
if missing_outputs:
    raise RuntimeError(
        "Faltan outputs del Notebook 57:\n- "
        + "\n- ".join(missing_outputs)
    )

final_status = (
    "APPROVED_FOR_NOTEBOOK_58"
    if approved
    else "PREFLIGHT_REVIEW_REQUIRED"
)
print({
    "status": final_status,
    "approved": approved,
    "nextNotebook": 58 if approved else None,
    "humanReviewRequired": True,
    "notClinicalDiagnosis": True,
    "officialTestAccessed": False,
})


{'status': 'APPROVED_FOR_NOTEBOOK_58', 'approved': True, 'nextNotebook': 58, 'humanReviewRequired': True, 'notClinicalDiagnosis': True, 'officialTestAccessed': False}
